# Notebook 01: Data Exploration

Explore the conjunctiva image dataset and tabular patient metadata for anemia severity screening.

**Target classes:** Normal, Mild, Moderate, Severe

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

# Configure display
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", None)


## 2. Load Metadata

In [ ]:
# Paths (adjust BASE_DIR if running from a different location)
BASE_DIR = Path("..")          # project root when running from project/notebooks/
META_PATH = BASE_DIR / "data" / "metadata.csv"

df = pd.read_csv(META_PATH)
print(f"Dataset shape: {df.shape}")
df.head()


## 3. Verify Missing Values

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing: {missing.sum()}")


## 4. Class Distribution

In [ ]:
print("Class distribution:")
print(df["diagnosis"].value_counts())

fig, ax = plt.subplots(figsize=(8, 5))
order = ["Normal", "Mild", "Moderate", "Severe"]
colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]
counts = df["diagnosis"].value_counts().reindex(order, fill_value=0)
ax.bar(counts.index, counts.values, color=colors, edgecolor="black", linewidth=0.7)
ax.set_title("Anemia Severity Class Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Severity Class")
ax.set_ylabel("Number of Samples")
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("../models/saved_models/class_distribution.png", bbox_inches="tight")
plt.show()


## 5. Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of age (in months)
axes[0].hist(df["age"], bins=30, color="#3498db", edgecolor="black", linewidth=0.5)
axes[0].set_title("Age Distribution (Months)")
axes[0].set_xlabel("Age (Months)")
axes[0].set_ylabel("Frequency")

# Age by diagnosis
order = ["Normal", "Mild", "Moderate", "Severe"]
sns.boxplot(x="diagnosis", y="age", data=df, order=order,
            palette=["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], ax=axes[1])
axes[1].set_title("Age Distribution by Severity Class")
axes[1].set_xlabel("Severity Class")
axes[1].set_ylabel("Age (Months)")
plt.tight_layout()
plt.savefig("../models/saved_models/age_distribution.png", bbox_inches="tight")
plt.show()


## 6. Gender Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_counts = df["gender"].value_counts()
axes[0].pie(gender_counts.values, labels=gender_counts.index, autopct="%1.1f%%",
            colors=["#3498db", "#e91e8c"], startangle=90)
axes[0].set_title("Overall Gender Split")

# Gender vs Diagnosis
ct = pd.crosstab(df["diagnosis"], df["gender"])
ct = ct.reindex(["Normal", "Mild", "Moderate", "Severe"])
ct.plot(kind="bar", ax=axes[1], color=["#3498db", "#e91e8c"], edgecolor="black")
axes[1].set_title("Severity vs Gender")
axes[1].set_xlabel("Severity Class")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 7. Display Sample Images

In [ ]:
def show_sample_images(df, n_per_class=3):
    """Display sample conjunctiva images for each severity class."""
    classes = ["Normal", "Mild", "Moderate", "Severe"]
    colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]
    fig, axes = plt.subplots(len(classes), n_per_class,
                             figsize=(n_per_class * 4, len(classes) * 3))
    for row_idx, (cls, col) in enumerate(zip(classes, colors)):
        subset = df[df["diagnosis"] == cls].sample(
            min(n_per_class, len(df[df["diagnosis"] == cls])), random_state=42)
        for col_idx, (_, sample) in enumerate(subset.iterrows()):
            ax = axes[row_idx, col_idx]
            img_path = BASE_DIR / sample["image_path"]
            if img_path.exists():
                img = mpimg.imread(str(img_path))
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, "Image\nnot found",
                        ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{cls}", fontsize=10, color=col, fontweight="bold")
            ax.axis("off")
    plt.suptitle("Sample Conjunctiva Images by Severity Class",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("../models/saved_models/sample_images.png", bbox_inches="tight")
    plt.show()

show_sample_images(df)


## 8. Summary Statistics

In [ ]:
print("=== Dataset Summary ===")
print(f"Total images: {len(df)}")
print(f"\nClass distribution:")
for cls in ["Normal", "Mild", "Moderate", "Severe"]:
    count = (df["diagnosis"] == cls).sum()
    pct = count / len(df) * 100
    print(f"  {cls:10s}: {count:3d} ({pct:.1f}%)")
print(f"\nAge statistics (months):")
print(df["age"].describe().round(2))
print(f"\nGender distribution:")
print(df["gender"].value_counts())
